<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/rf_labeled_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Labeled example receptive fields across three recording scales

A self-contained gallery of well-formed example receptive fields (RFs) for each recording technique — Neuropixels spikes, mesoscope two-photon ΔF/F, and SLAP2 dendritic glutamate — with **every example labeled by a compact, human-legible provenance ID** so anyone can locate the exact source unit/ROI without a lookup table.

**ID scheme** (fields joined by `·`):

- `NP·<subject>·<date>·<area+layer>·pr<Probe>·d<depth µm>·u<unit>` — Neuropixels
- `MS·<subject>·<date>·<area>·<plane-depth µm>·roi<id>·(<x,y> µm)` — Mesoscope soma
- `SL·<subject>·<date>·<DMD>·roi<id>·(<x,y> px)` — SLAP2 dendritic ROI (no CCF area — glutamate)

Examples are picked by **2-D Gaussian fit quality** (not peak SNR, which is biased toward spiky one-pixel maps); the colour scale is a diverging LUT centred at zero with a black half-max contour. All logic lives in `_rf_labeled_cells.py` (committed alongside this notebook), so the notebook stays short and the gallery regenerates from one clone + Run-All.

> **Note.** This notebook streams three NWB assets from the DANDI archive; the mesoscope arm (per-ROI RF maps across two imaging planes) is the slow step (~3–6 min). Sessions were chosen for RF yield: ecephys sub-830851, mesoscope sub-832700, SLAP2 sub-796630 (2025-10-01, DMD1).

In [ ]:
#@title Install dependencies
!pip -q install pynwb h5py remfile requests pandas numpy scipy matplotlib

In [ ]:
#@title Load the extraction + gallery helpers
# In Colab, fetch the repo first so the helper module is importable:
# !git clone -q https://github.com/maierav/ai_oscp_neuro.git && %cd ai_oscp_neuro/notebooks
import sys; sys.path.insert(0, '.')
import importlib.util
spec = importlib.util.spec_from_file_location('rfl', '_rf_labeled_cells.py')
rfl = importlib.util.module_from_spec(spec); spec.loader.exec_module(rfl)
print('helpers loaded')

In [ ]:
#@title Neuropixels — RFs + provenance IDs (sub-830851)
ece = rfl.ecephys_rf()
ece['pick'][['uid','area','probe','depth_um','fwhm','r2']].round(2)

In [ ]:
#@title Mesoscope 2p — RFs + provenance IDs (sub-832700, VISp+VISl planes)
meso = rfl.mesoscope_rf()
meso['pick'][['uid','area','plane_depth_um','roi','cx_um','cy_um','fwhm','r2']].round(2)

In [ ]:
#@title SLAP2 — RFs + provenance IDs (sub-796630, DMD1)
slap = rfl.slap2_rf()
slap['pick'][['uid','dmd','roi','cx_px','cy_px','fwhm','r2']].round(2)

In [ ]:
#@title The gallery — labeled example RFs across all three scales
import os
specs = [ece, meso, slap]
fig = rfl.render_gallery(specs, savepath='../figures/rf_labeled_examples.png')
rfl.id_table(specs).to_csv('../data/rf_labeled_examples.csv', index=False)
fig

## Takeaway

All three techniques resolve well-formed, retinotopically-plausible RFs (FWHM ~15–40°, R² > 0.5), and each example is traceable to its source by a single compact ID — technique, animal, session date, anatomy, and within-session position at a glance. The full provenance table is written to `data/rf_labeled_examples.csv`.